# Solving the "LOST IN THE MIDDLE" Problem in RAG Systems

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_628\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
pdf_path1 = r'D:\AI Engineering\RAG-LAB\Re-Ranking\Agentic_Design_Patterns.pdf'
pdf_path2 = r'D:\AI Engineering\RAG-LAB\Re-Ranking\dokumen.pub_generative-deep-learning-teaching-machines-to-paint-write-compose-and-play-1492041947-978-1492041948.pdf'

In [4]:
loader_1  = PyPDFLoader(pdf_path1)
loader_2  = PyPDFLoader(pdf_path2)

In [5]:
document_1 = loader_1.load()
document_2 = loader_2.load()

for doc in document_2:
    doc.page_content = doc.page_content.encode("utf-16", "surrogatepass").decode("utf-16", "replace")

In [6]:
print(len(document_1))
print(len(document_2))

482
330


## create the text splitter for chunking

In [7]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=300)

In [9]:
text_1 = text_splitter.split_documents(document_1)
text_2 = text_splitter.split_documents(document_2)

In [10]:
print(len(text_1))
print(len(text_2))

1625
724


## Load the Embedding Model to Conver the Data into Vector

In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

google_embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [12]:
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "all-MiniLM-L6-v2"
model_path = Path.cwd() / "Re-Ranking" / "embedding_model" / model_name
if not model_path.is_dir():
    model_path = Path.cwd() / "embedding_model" / model_name
if not model_path.is_dir():
    raise FileNotFoundError(f"Embedding model directory not found: {model_path}")

local_embeddings = HuggingFaceEmbeddings(
    model_name=str(model_path),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

d:\AI Engineering\RAG-LAB\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2307.66it/s]


## Now ingest the Data into the Chroma Database

In [13]:
from langchain_community.vectorstores import Chroma
import chromadb

In [14]:
CURRENT_DIR = os.path.dirname(os.path.abspath("."))

In [15]:
DB_DIR = os.path.join(CURRENT_DIR, "/vector_embeddings")

In [16]:
client_settings = chromadb.config.Settings(
    is_persistent=True,
    persist_directory=DB_DIR,
    anonymized_telemetry=False,
)

In [17]:
document_1_vectorstore = Chroma.from_documents(text_1,
                                       local_embeddings,
                                       client_settings=client_settings,
                                       collection_name="Agentic-Design-Patterns",
                                       collection_metadata={"hnsw":"cosine"},
                                       persist_directory="/store/Agentic-Design-Patterns")

In [18]:
document_2_vectorstore = Chroma.from_documents(text_2,
                                       google_embeddings,
                                       client_settings=client_settings,
                                       collection_name="generative-ai",
                                       collection_metadata={"hnsw":"cosine"},
                                       persist_directory="/store/generative-ai"
                                    )

## Now Crearte a Retriever

In [19]:
retriever_1 = document_1_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})
retriever_2 = document_2_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

## Let's Merge both Retriever

## It is also called lord of retriever(LOTR)

In [20]:
from langchain_classic.retrievers.merger_retriever import MergerRetriever

In [21]:
lotr = MergerRetriever(retrievers=[retriever_1, retriever_2])

In [25]:
for chunks in lotr.invoke("Explain the RAG Design Pattern?"):
    print(chunks.page_content)

recognize
 
that
 
they
 
refer
 
to
 
the
 
same
 
thing
 
and
 
would
 
consider
 
them
 
to
 
be
 
highly
 
similar.
 
This
 
is
 
because
 
their
 
embeddings
 
would
 
be
 
very
 
close
 
in
 
the
 
vector
 
space,
 
indicating
 
a
 
small
 
semantic
 
distance.
 
This
 
is
 
the
 
"smart
 
search"
 
that
 
allows
 
RAG
 
to
 
find
 
relevant
 
information
 
even
 
when
 
the
 
user's
 
wording
 
doesn't
 
exactly
 
match
 
the
 
text
 
in
 
the
 
knowledge
 
base.
 
 
Fig.1:
 
RAG
 
Core
 
Concepts:
 
Chunking,
 
Embeddings,
 
and
 
Vector
 
Database
 
Chunking
 
of
 
Documents:
 
Chunking
 
is
 
the
 
process
 
of
 
breaking
 
down
 
large
 
documents
 
into
 
smaller,
 
more
 
manageable
 
pieces,
 
or
 
"chunks."
 
For
 
a
 
RAG
 
system
 
to
 
work
 
efficiently,
 
it
 
cannot
 
feed
 
entire
 
large
 
documents
 
into
 
the
 
LLM.
 
Instead,
 
it
 
processes
 
these
 
smaller
 
chunks.
 
The
 
way
 
documents
 
are
 
chunked
 
is
 
important
 
for
 
preserving
 
the
Figure 5

### this result is too much messy now lets refine it according to the question and overcome the situation of lost in middle

## Now After understanding step by step it create a pipeline for LLM

In [26]:
from langchain_community.document_transformers import (
    EmbeddingsClusteringFilter,
    EmbeddingsRedundantFilter,
    LongContextReorder
)
from langchain_classic.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_classic.retrievers import ContextualCompressionRetriever

In [27]:
from re import search

filter = EmbeddingsRedundantFilter(embeddings=local_embeddings)

reordering = LongContextReorder()

pipeline = DocumentCompressorPipeline(transformers=[filter, reordering])

compression_retriever_reordered = ContextualCompressionRetriever(
    base_compressor=pipeline, base_retriever=lotr,search_kwargs={"k": 3, "include_metadata": True}
)

In [28]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

In [29]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.7-flash",
    temperature=0,
)

In [30]:
qa = RetrievalQA.from_chain_type(
      llm=llm,
      chain_type="stuff",
      retriever = compression_retriever_reordered,
      return_source_documents = True
)

In [35]:
query ="Explain the parallelization Design Pattern?"
results = qa(query)
print("result is as follow:-", "\n")
print(results['result'])
print("\n")
print("source documents is as follow:-", "\n")
print(results["source_documents"])

result is as follow:- 

Based on the provided context, the **Parallelization design pattern** is a workflow pattern designed to execute multiple independent tasks or operations concurrently rather than sequentially (one after another). 

Here is an explanation of its key aspects:

### 1. Core Purpose and Mechanism
* **Reduces Total Execution Time:** By running independent sub-tasks simultaneously, the pattern drastically cuts down overall workflow time, improving efficiency and responsiveness.
* **Orchestration:** A main process or orchestrator invokes several sub-tasks that run in parallel, waits for all of them to complete, and then gathers or synthesizes the results before moving to the next step.

### 2. When to Use It (Rule of Thumb)
Use this pattern whenever a workflow contains independent operations that can execute at the same time without relying on each other's intermediate outputs. Typical scenarios include:
* Fetching data from multiple APIs or external services simultaneou